# Revenue Management EDA — Reusable Company Workflow (v2.1 / GitHub-to-Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlosjostemperor/rental-revenue-decision-support/blob/main/Revenue_Management_EDA.ipynb)

**1-click startup:** open the notebook from the GitHub link above, then run section 0. In Colab, the setup cell automatically clones the **latest code** from `main` (or updates the existing clone). There is **no project ZIP upload**. This is **one-way code sync (GitHub → Colab on setup)**, not automatic saving back to GitHub.

**Purpose:** an auditable, repeatable pricing-position study for one property, built on four PriceLabs exports (Competitor Calendar, Bookings Report, two Report Builder exports).

**Workflow:** sync code → provide four CSV exports → configure → run → review → export evidence.

All analytical logic lives in the tested `rm_analytics` package; the notebook configures, displays and exports. The public GitHub repository contains no real PriceLabs exports. Do not save business data, executed outputs, or operational analysis files into a public GitHub notebook.

**Definitions:** competitor advertised rates ≠ PriceLabs market ADR ≠ realized booking ADR. Report Builder Stay Date Range ≠ Booked Date filter. Competitor unavailability ≠ confirmed occupancy. Pricing comparisons do not prove causality.


## 0 | Connect GitHub and Colab (automatic)

Run this cell first. In **Google Colab**, it downloads the current `main` branch of the public repository into `/content/rental-revenue-decision-support`, then imports `rm_analytics` from that folder. Re-running the cell fetches updates. Other files in the Colab virtual machine are temporary; project inputs in `data/input/` are ignored by Git and never uploaded automatically.

**Important:** refreshing the notebook page alone does not update code already loaded into the Python runtime; run this setup cell again. GitHub → Colab is one-way here; no automatic pushes or credentials.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPOSITORY = "https://github.com/carlosjostemperor/rental-revenue-decision-support.git"
BRANCH = "main"


def _find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "rm_analytics" / "__init__.py").is_file():
            return candidate
    return None


try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/rental-revenue-decision-support")
    if (PROJECT_ROOT / ".git").is_dir():
        print("Syncing the latest code from GitHub (main)...")
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    elif PROJECT_ROOT.exists():
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a Git repository. "
            "Choose a clean Colab runtime to avoid overwriting local data."
        )
    else:
        print("Cloning the public repository from GitHub...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPOSITORY, str(PROJECT_ROOT)],
            check=True,
        )
    revision = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
        text=True,
    ).strip()
    print(f"GitHub version: {revision} ({BRANCH})")
else:
    PROJECT_ROOT = _find_project_root(Path.cwd())
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Run this notebook from the project folder containing rm_analytics/. "
            "For one-click setup, open it from GitHub in Google Colab."
        )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

# When the setup cell is re-run, do not keep older imported rm_analytics modules.
for module_name in list(sys.modules):
    if module_name == "rm_analytics" or module_name.startswith("rm_analytics."):
        del sys.modules[module_name]

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import rm_analytics
from rm_analytics import AnalysisConfig, run_analysis, export_run
from rm_analytics.charts import build_figures

pd.set_option("display.max_colwidth", 160)
print("Project:", PROJECT_ROOT, "| rm_analytics", rm_analytics.__version__, "| pandas", pd.__version__)
print("To upload real reports, use optional section 0B. To use synthetic data, skip it.")


## 0B | Upload the four PriceLabs CSVs (optional, Colab only)

To analyze a **real property**, set `UPLOAD_CSVS = True` below and run this cell. You will be asked **four times**, once for each export, and you may select files with their original PriceLabs filenames. They are saved privately in this Colab runtime as `data/input/competitor_calendar.csv`, `bookings.csv`, `report_builder_historical.csv`, and `report_builder_forward.csv`. The notebook **does not upload them to GitHub**. For the example dataset, leave `UPLOAD_CSVS = False`.

Bookings Report should include **all bookings for the target property** so the stay-date reconciliation can be performed. Confirm the historical and forward date ranges in the two Report Builder exports.


In [ ]:
UPLOAD_CSVS = False  # Change to True to upload four real exports from your computer.

if UPLOAD_CSVS:
    if not IN_COLAB:
        raise RuntimeError("This upload helper is designed for Google Colab. For local use, copy the CSVs to data/input/.")
    from google.colab import files
    targets = [
        ("Competitor Calendar", "competitor_calendar.csv"),
        ("Bookings Report (all reservations for this property)", "bookings.csv"),
        ("Report Builder — HISTORICAL stay dates", "report_builder_historical.csv"),
        ("Report Builder — FORWARD stay dates", "report_builder_forward.csv"),
    ]
    collected = {}
    for label, filename in targets:
        print(f"\nSelect exactly one CSV for: {label}")
        incoming = files.upload()
        if len(incoming) != 1:
            raise ValueError(f"Expected exactly one CSV for {label}; none have been saved.")
        original_name, raw_bytes = next(iter(incoming.items()))
        if not original_name.lower().endswith(".csv"):
            raise ValueError(f"Expected a .csv file for {label}, got {original_name!r}; none have been saved.")
        if not raw_bytes:
            raise ValueError(f"The {label} CSV was empty; none have been saved.")
        collected[filename] = raw_bytes
    input_dir = PROJECT_ROOT / "data" / "input"
    input_dir.mkdir(parents=True, exist_ok=True)
    for filename, contents in collected.items():
        (input_dir / filename).write_bytes(contents)
    print("\nFour exports uploaded to:", input_dir)
    print("They remain in this Colab runtime and will not be committed to GitHub.")
else:
    print("Upload skipped. To use real exports, set UPLOAD_CSVS = True and run this cell.")


## 1 | Configure this run

* `property_name`: exact listing name as in the exports (an exact match is preferred; a partial match must be unique).
* `booking_start/end`: **Booked Date** window (reservation creation). Inclusive, any time of day.
* `historical_stay_*` and `forward_*`: the **Stay Date Ranges you selected in Report Builder**. They are cross-checked against the Bookings report; mismatches are reported.
* `data_dir`: `data/input` for real work (use optional section 0B to upload CSVs in Colab). To try the tool, run the next cell's optional line to create synthetic example data and use `data/example_synthetic`.
* Attestations are recorded in the manifest; set them to `True` only after doing the check.

In [ ]:
# Optional: create the synthetic example dataset (no real data) and try the tool with it.
# from rm_analytics.synthetic import generate_synthetic_dataset; generate_synthetic_dataset('data/example_synthetic')

DATA_DIR = os.environ.get('RMA_DATA_DIR', 'data/input')
IS_SYNTHETIC = 'synthetic' in DATA_DIR

cfg = AnalysisConfig(
    property_name=os.environ.get('RMA_PROPERTY', 'YOUR LISTING NAME'),
    booking_start='2026-10-11',
    booking_end='2026-11-02',
    historical_stay_start='2026-10-11',
    historical_stay_end='2026-11-02',
    forward_start='2026-11-03',
    forward_end='2027-01-31',
    data_dir=DATA_DIR,
    min_active_comps=5,             # minimum available comps for a date to be compared
    calendar_date_format='auto',    # or e.g. '%d/%m/%Y' if detection reports ambiguity
    bookings_date_format='auto',
    duplicate_policy='warn',        # 'drop' removes the second listing of a near-identical pair
    exclude_comps=(),               # comp names to exclude after review
    analyst='',                     # your name
    reviewer='',
    attest_report_periods_verified=False,
    attest_comp_set_reviewed=False,
    attest_storage_approved=False,
    is_synthetic_example=IS_SYNTHETIC,
)
print('Input files:'); [print(' ', k, '->', p) for k, p in cfg.input_paths().items()]

## 2 | Run: load, validate, compute, audit
Structural problems (unparseable money/dates, unknown booking statuses, ambiguous listing names, calendar gaps) stop the run with a clear message. Fix the input; do not bypass the check.

In [ ]:
result = run_analysis(cfg)
audit_df = result.audit.to_frame()
print('Subject listing:', result.calendar.target)
print('Comps used:', len(result.comps), '| dropped as duplicates:', result.dropped_comps)
print('Audit:', result.audit.counts())
display(audit_df[audit_df.status != 'PASS'])
print('DECISION STATUS:', result.audit.decision_status)
if result.audit.decision_status != 'VALIDATED':
    print('REVIEW REQUIRED: inspect audit warnings and document resolution before relying on conclusions.')


## 3 | Executive metrics
Pricing metrics compare **advertised** rates on our **available** dates in the forward window. The confidence interval accounts for within-week correlation. Compare the bias-sensitivity row with the main gap: a large difference means the availability filter is driving the result.

In [ ]:
display(result.summary_table.round(2))

## 4 | ADR, occupancy and RevPAR versus PriceLabs market
RevPAR proxy = ADR × occupancy. It answers whether a lower rate was compensated by higher occupancy. Occupancy definitions may differ between Adjusted and Market occupancy; verify in PriceLabs.

In [ ]:
display(result.market.T)
print('Reconciliation of bookings (allocated to stay dates) vs Report Builder:')
display(result.reconciliation)

## 5 | Daily detail

In [ ]:
display(result.band_by_day_group)
cols = ['our_price','n_comps','comp_p25','comp_median','comp_p75','comp_std','gap_vs_median_pct','gap_vs_all_priced_median_pct','our_min_stay','comp_median_min_stay','price_band','exclusion_reason']
display(result.positions[cols].round(2).head(20))
if len(result.duplicates):
    print('Probable duplicate homes:'); display(result.duplicates)
print('Minimum-stay sensitivity:')
from rm_analytics.export import _minimum_stay_sensitivity
display(_minimum_stay_sensitivity(result).round(2))


## 6 | Charts

In [ ]:
figures = build_figures(result)
plt.show()

## 7 | Export evidence
Creates `output/<run_id>/` with tables, minimum-stay sensitivity, `REVIEW_SUMMARY.md`, charts, audit and `run_manifest.json` (input and output SHA-256, config, code and environment versions). Runs are **VALIDATED** (zero audit warnings/failures), **REVIEW_REQUIRED** (one or more WARN), or **FAILED** (one or more FAIL). Review-required runs are exported to folders ending in `_REVIEW_REQUIRED` and may not be treated as validated. All price changes require separate approval. In Colab, download the folder before the session ends.

In [ ]:
run_dir = export_run(result, figures)
print('Exported to', run_dir)
for p in sorted(run_dir.iterdir()):
    print(' -', p.name)
if result.audit.has_failures:
    raise RuntimeError(f'Audit FAIL checks present; run exported as FAILED for traceability: {run_dir.name}')
print('DECISION STATUS:', result.audit.decision_status)
if result.audit.decision_status == 'REVIEW_REQUIRED':
    print('Not yet validated. Read REVIEW_SUMMARY.md and resolve or document review items.')


## 8 | Reviewer checklist and interpretation

1. Confirm property identity, date windows and that all four exports come from the same PriceLabs filters.
2. Resolve WARNs, or document your acceptance and rationale in the review record; the generated status remains REVIEW_REQUIRED until rerun with all checks passing. Pay attention to reconciliation, minimum-stay comparability and bias sensitivity.
3. Spot-check five dates against the PriceLabs UI, including a weekend and a low-comp date.
4. Interpret historical and forward market comparisons separately. Check RevPAR, not only ADR or occupancy.
5. Consider alternative explanations: listing age and review count, launch discounts, search ranking, minimum stays, cleaning fees, seasonality.
6. Any rate change is a **pricing test** with a hypothesis, dates, magnitude, monitoring metric and rollback rule, approved separately.

**Conclusion framework:** Observation → Evidence → Alternative explanations → Proposed test → Risk/monitoring → Approval status.